In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

train = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

print(f'train: {train.shape}, test: {test.shape}, layout: {layout.shape}')

train: (250000, 94), test: (50000, 93), layout: (300, 15)


## 1. 전처리 - Scenario-level Aggregation

In [5]:
# Scenario-level 통계 (train 기준으로 생성 → test에 매핑)
agg_features = train.groupby('scenario_id').agg(
    sc_battery_mean_avg   = ('battery_mean', 'mean'),
    sc_low_battery_max    = ('low_battery_ratio', 'max'),
    sc_congestion_avg     = ('congestion_score', 'mean'),
    sc_order_inflow_avg   = ('order_inflow_15m', 'mean'),
    sc_order_inflow_max   = ('order_inflow_15m', 'max'),
    sc_robot_charging_avg = ('robot_charging', 'mean'),
).reset_index()

# Layout merge (building_age_years 제외 - 불필요 컬럼)
layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

# Train merge
train = train.merge(agg_features, on='scenario_id', how='left')
train = train.merge(layout_clean, on='layout_id', how='left')

# Test merge
test = test.merge(agg_features, on='scenario_id', how='left')
test = test.merge(layout_clean, on='layout_id', how='left')

print(f'train: {train.shape}, test: {test.shape}')
print('중복 컬럼:', [c for c in train.columns if '_x' in c or '_y' in c])

train: (250000, 113), test: (50000, 112)
중복 컬럼: []


## 2. 전처리 - 결측치 처리

In [6]:
TARGET = 'avg_delay_minutes_next_30m'

# 수치형 컬럼 분리
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_no_target = [c for c in numeric_cols if c != TARGET]
numeric_cols_test = test.select_dtypes(include=[np.number]).columns.tolist()

# 결측 indicator 생성 (결측 자체가 정보)
missing_cols = [col for col in numeric_cols_no_target if train[col].isnull().any()]
for col in missing_cols:
    train[f'{col}_missing'] = train[col].isnull().astype(int)
    test[f'{col}_missing']  = test[col].isnull().astype(int)

print(f'결측 indicator 추가: {len(missing_cols)}개')

# Median imputation (train fit → test transform, leakage 방지)
imputer = SimpleImputer(strategy='median')
train[numeric_cols_no_target] = imputer.fit_transform(train[numeric_cols_no_target])
test[numeric_cols_test]       = imputer.transform(test[numeric_cols_test])

print(f'결측치 잔여 - train: {train.isnull().sum().sum()}, test: {test.isnull().sum().sum()}')
print(f'train: {train.shape}, test: {test.shape}')

결측 indicator 추가: 86개
결측치 잔여 - train: 0, test: 0
train: (250000, 199), test: (50000, 198)


## 3. Feature Engineering

In [7]:
def add_features(df):
    new_feats = pd.DataFrame(index=df.index)
    
    # Battery-Robot 상호작용
    new_feats['battery_robot_stress']  = df['low_battery_ratio'] * df['robot_charging']
    new_feats['charge_pressure']       = df['charge_queue_length'] / (df['robot_idle'] + 1)
    new_feats['effective_robot_ratio'] = df['robot_active'] / (df['robot_active'] + df['robot_charging'] + 1)
    
    # 주문-용량 압박
    new_feats['order_robot_ratio'] = df['order_inflow_15m'] / (df['robot_active'] + 1)
    new_feats['order_complexity']  = df['order_inflow_15m'] * df['avg_items_per_order']
    new_feats['urgent_pressure']   = df['urgent_order_ratio'] * df['order_inflow_15m']
    
    # Layout 연계
    new_feats['robots_per_area']     = df['robot_total'] / df['floor_area_sqm']
    new_feats['charger_sufficiency'] = df['charger_count'] / df['robot_total']
    new_feats['pack_capacity_ratio'] = df['pack_utilization'] / (df['pack_station_count'] + 1)
    
    # Shift hour 주기성
    new_feats['shift_hour_sin']  = np.sin(2 * np.pi * df['shift_hour'] / 24)
    new_feats['shift_hour_cos']  = np.cos(2 * np.pi * df['shift_hour'] / 24)
    new_feats['late_shift_flag'] = (df['shift_hour'] >= 18).astype(int)
    
    return pd.concat([df, new_feats], axis=1).copy()

train = add_features(train)
test  = add_features(test)

print(f'train: {train.shape}, test: {test.shape}')

train: (250000, 211), test: (50000, 210)


## 4. 인코딩 & 불필요 컬럼 제거

In [8]:
# layout_type 인코딩
le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'])
test['layout_type']  = le.transform(test['layout_type'])

# ID, layout_id, scenario_id 제거
drop_cols = ['ID', 'layout_id', 'scenario_id']
train = train.drop(columns=drop_cols)
test  = test.drop(columns=drop_cols)

print(f'train: {train.shape}, test: {test.shape}')
print('object 컬럼 잔여:', train.select_dtypes(include='object').columns.tolist())

train: (250000, 208), test: (50000, 207)
object 컬럼 잔여: []


## 5_. sc_delay_mean_loo 추가

In [10]:
# sc_delay_mean_loo 생성 (Leave-One-Out, target leakage 방지)
scenario_id_train = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')['scenario_id']
scenario_id_test  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['scenario_id']

train['scenario_id'] = scenario_id_train.values
test['scenario_id']  = scenario_id_test.values

# train: 자기 자신 제외한 시나리오 평균
train['sc_delay_mean_loo'] = (
    train.groupby('scenario_id')[TARGET]
    .transform(lambda x: (x.sum() - x) / (x.count() - 1))
)

# test: train 시나리오별 전체 평균 매핑
sc_delay_map = train.groupby('scenario_id')[TARGET].mean()
test['sc_delay_mean_loo'] = test['scenario_id'].map(sc_delay_map)

# scenario_id 다시 제거
train = train.drop(columns=['scenario_id'])
test  = test.drop(columns=['scenario_id'])

print(train['sc_delay_mean_loo'].describe())
print(f'train: {train.shape}, test: {test.shape}')

count    250000.000000
mean         18.962296
std          21.794810
min           0.000000
25%           4.611397
50%           9.561479
75%          27.503643
max         235.771676
Name: sc_delay_mean_loo, dtype: float64
train: (250000, 209), test: (50000, 208)


## 5. 모델 학습 (Best: model5, num_leaves=255)

In [11]:
X = train.drop(columns=[TARGET])
y = np.log1p(train[TARGET])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_loo = lgb.LGBMRegressor(
    n_estimators     = 5000,
    learning_rate    = 0.05,
    num_leaves       = 255,
    max_depth        = -1,
    min_child_samples= 20,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 0.1,
    random_state     = 42,
    n_jobs           = -1
)

model_loo.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)]
)

joblib.dump(model_loo, 'C:/Users/user/dacon/Smart_Logistics/models/lgbm_model_loo.pkl')

y_pred = model_loo.predict(X_val)
rmse_orig = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(y_pred)))
print(f'Validation RMSE (original scale): {rmse_orig:.4f}')

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.051131 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25330
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 208
[LightGBM] [Info] Start training from score 2.423409
Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.206603
Early stopping, best iteration is:
[754]	valid_0's l2: 0.206341
Validation RMSE (original scale): 17.7637


In [12]:
test_id = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['ID']

y_test_pred      = model_loo.predict(test)
y_test_pred_orig = np.expm1(y_test_pred)

submission = pd.DataFrame({
    'ID'                        : test_id,
    'avg_delay_minutes_next_30m': y_test_pred_orig
})

submission.to_csv('C:/Users/user/dacon/Smart_Logistics/results/submission_model_loo.csv', index=False)
print(submission.head())

            ID  avg_delay_minutes_next_30m
0  TEST_000000                    1.255168
1  TEST_000001                    1.015929
2  TEST_000002                    1.690927
3  TEST_000003                    1.495015
4  TEST_000004                    1.156778


In [14]:
train_scenarios = set(pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')['scenario_id'])
test_scenarios  = set(pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['scenario_id'])

print(f'train 시나리오 수: {len(train_scenarios)}')
print(f'test 시나리오 수:  {len(test_scenarios)}')
print(f'겹치는 시나리오:   {len(train_scenarios & test_scenarios)}')

train 시나리오 수: 10000
test 시나리오 수:  2000
겹치는 시나리오:   0
